# Обучение YOLOv8 на чертежах ЕСКД (ООО «БВК» / ЮУрГУ)
Данный блокнот обучает нейросеть детекции элементов чертежа (поле видов, основная надпись/штамп ГОСТ 2.104, технические требования) на бесплатной видеокарте GPU (NVIDIA T4 / L4).

### Инструкция:
1. В меню выберите: **Среда выполнения -> Сменить среду выполнения -> T4 GPU**.
2. Нажмите **Среда выполнения -> Выполнить все** (`Ctrl + F9`).

In [ ]:
# 1. Проверка доступности GPU
!nvidia-smi

In [ ]:
# 2. Клонирование репозитория проекта
import os
if not os.path.exists('pricing_of_a_unique_cast_product'):
    # Для приватного репозитория: можно добавить токен в Secrets Colab (имя GITHUB_TOKEN)
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/yaroslikch-glitch/pricing_of_a_unique_cast_product.git
    except Exception:
        !git clone https://github.com/yaroslikch-glitch/pricing_of_a_unique_cast_product.git

%cd pricing_of_a_unique_cast_product
!git pull origin main

In [ ]:
# 3. Установка библиотек
!pip install -q ultralytics pymupdf matplotlib pillow

In [ ]:
# 4. Запуск обучения YOLO на GPU
from ultralytics import YOLO

# Используем текущие обученные веса eskd_yolo_best.pt или yolov8n.pt
weights = 'eskd_yolo_best.pt' if os.path.exists('eskd_yolo_best.pt') else 'yolov8n.pt'
print(f'Инициализация модели: {weights}')
model = YOLO(weights)

# Обучение 30 эпох на 1 000 чертежах ЕСКД с GPU ускорением
results = model.train(
    data='dataset_yolo/dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,  # NVIDIA GPU
    project='runs_colab',
    name='eskd_yolo_gpu',
    exist_ok=True,
    save=True,
    plots=True
)

In [ ]:
# 5. Визуализация графиков обучения и метрик (mAP50, Loss)
from IPython.display import Image, display

results_img = 'runs_colab/eskd_yolo_gpu/results.png'
if os.path.exists(results_img):
    print('=== Графики обучения (Loss & mAP) ===')
    display(Image(results_img))

val_img = 'runs_colab/eskd_yolo_gpu/val_batch0_pred.jpg'
if os.path.exists(val_img):
    print('=== Пример детекции на валидационных чертежах ===')
    display(Image(val_img))

In [ ]:
# 6. Скачивание готовых весов best.pt на локальный компьютер
from google.colab import files
import shutil

best_weights = 'runs_colab/eskd_yolo_gpu/weights/best.pt'
if os.path.exists(best_weights):
    print('Скачивание файла best.pt...')
    files.download(best_weights)
else:
    print('Файл весов не найден')